In [ ]:
#| default_exp cli

# cli

> Command-line interface for manhualizer.
>
> Entry point: `manhualizer` (configured in `[project.scripts]`).
>
> ```sh
> # Dead simple
> manhualizer story.txt
>
> # Common options
> manhualizer story.txt --model nanobanana --template noir -o my_comic/
>
> # Full config file
> manhualizer story.txt --config my_project.yml
>
> # Step commands
> manhualizer analyze-only story.txt
> manhualizer storyboard-only story.txt
> manhualizer render-only story.txt
> ```

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
from pathlib import Path
from typing import Optional

import typer
from rich.console import Console
from typing_extensions import Annotated

from manhualizer.render import list_models

app = typer.Typer(
    name="manhualizer",
    help="Convert a story into a manhua comic. Pass a story text file to get started.",
    add_completion=False,
    pretty_exceptions_enable=False,
)
_console = Console()

In [ ]:
#| export
def _load(config_path, model, template, output_dir, fmt, width, height, aspect_ratio,
          validate, no_resume, lora):
    """Build a PipelineConfig from CLI args, loading .env and config file first."""
    from dotenv import load_dotenv
    load_dotenv()

    from manhualizer.config import load_config
    overrides = {
        "model": model,
        "template": template,
        "output_dir": output_dir,
        "format": fmt,
        "width": width,
        "height": height,
        "aspect_ratio": aspect_ratio,
        "run_validation": True if validate else None,
        "resume": False if no_resume else None,
    }
    cfg = load_config(config_path, **{k: v for k, v in overrides.items() if v is not None})

    # LoRA from CLI: append to existing config loras
    if lora:
        from manhualizer.config import LoRAConfig
        cfg.renderer.loras.extend(
            LoRAConfig(path=str(p)) for p in lora
        )
    return cfg

In [ ]:
#| export
# Option help strings (shared for consistency, options defined inline per command)
_MODELS_STR = f"Image model. One of: {', '.join(list_models())}"

In [ ]:
#| export
@app.command()
def convert(
    story:        Annotated[Path,           typer.Argument(help="Path to the story text file.")],
    config:       Annotated[Optional[Path], typer.Option("--config", "-c", help="Path to manhualizer.yml.")] = None,
    model:        Annotated[Optional[str],  typer.Option("--model", "-m", help=_MODELS_STR)] = None,
    template:     Annotated[Optional[str],  typer.Option("--template", "-t", help="Prompt template name.")] = None,
    output:       Annotated[Optional[str],  typer.Option("--output", "-o", help="Output directory.")] = None,
    fmt:          Annotated[Optional[str],  typer.Option("--format", "-f", help="Image format: png, jpg, webp.")] = None,
    width:        Annotated[Optional[int],  typer.Option("--width", "-W", help="Output width in pixels.")] = None,
    height:       Annotated[Optional[int],  typer.Option("--height", "-H", help="Output height in pixels.")] = None,
    aspect_ratio: Annotated[Optional[str],  typer.Option("--aspect-ratio", "-r", help="Aspect ratio e.g. '9:16'.")] = None,
    validate:     Annotated[bool,           typer.Option("--validate", help="Run LLM validation step.")] = False,
    no_resume:    Annotated[bool,           typer.Option("--no-resume", help="Rerun all steps.")] = False,
    lora:         Annotated[Optional[list[Path]], typer.Option("--lora", "-l", help="LoRA path/URL (repeatable).")] = None,
):
    """Convert a story file into a manhua comic (full pipeline)."""
    cfg = _load(config, model, template, output, fmt, width, height,
                aspect_ratio, validate, no_resume, lora)
    from manhualizer.pipeline import run
    run(story, cfg)


@app.command(name="analyze-only")
def analyze_only(
    story:     Annotated[Path,           typer.Argument(help="Path to the story text file.")],
    config:    Annotated[Optional[Path], typer.Option("--config", "-c", help="Path to manhualizer.yml.")] = None,
    no_resume: Annotated[bool,           typer.Option("--no-resume", help="Rerun all steps.")] = False,
):
    """Run only the story analysis step (saves analysis.json)."""
    from dotenv import load_dotenv; load_dotenv()
    from manhualizer.config import load_config
    from manhualizer.pipeline import run_analyze_only
    cfg = load_config(config, resume=not no_resume)
    result = run_analyze_only(story, cfg)
    _console.print(f"[green]Analysis complete:[/green] {len(result.characters)} character(s), "
                   f"{len(result.locations)} location(s)")


@app.command(name="storyboard-only")
def storyboard_only(
    story:     Annotated[Path,           typer.Argument(help="Path to the story text file.")],
    config:    Annotated[Optional[Path], typer.Option("--config", "-c", help="Path to manhualizer.yml.")] = None,
    no_resume: Annotated[bool,           typer.Option("--no-resume", help="Rerun all steps.")] = False,
):
    """Run analysis + storyboard steps (saves analysis.json and storyboard.json)."""
    from dotenv import load_dotenv; load_dotenv()
    from manhualizer.config import load_config
    from manhualizer.pipeline import run_storyboard_only
    cfg = load_config(config, resume=not no_resume)
    result = run_storyboard_only(story, cfg)
    _console.print(f"[green]Storyboard complete:[/green] {result.total_panels} panel(s) across "
                   f"{len(result.scenes)} scene(s)")


@app.command(name="render-only")
def render_only(
    story:        Annotated[Path,           typer.Argument(help="Path to the story text file.")],
    config:       Annotated[Optional[Path], typer.Option("--config", "-c", help="Path to manhualizer.yml.")] = None,
    model:        Annotated[Optional[str],  typer.Option("--model", "-m", help=_MODELS_STR)] = None,
    fmt:          Annotated[Optional[str],  typer.Option("--format", "-f", help="Image format: png, jpg, webp.")] = None,
    width:        Annotated[Optional[int],  typer.Option("--width", "-W", help="Output width in pixels.")] = None,
    height:       Annotated[Optional[int],  typer.Option("--height", "-H", help="Output height in pixels.")] = None,
    aspect_ratio: Annotated[Optional[str],  typer.Option("--aspect-ratio", "-r", help="Aspect ratio e.g. '9:16'.")] = None,
    no_resume:    Annotated[bool,           typer.Option("--no-resume", help="Rerun all steps.")] = False,
    lora:         Annotated[Optional[list[Path]], typer.Option("--lora", "-l", help="LoRA path/URL (repeatable).")] = None,
):
    """Run only the render step (requires analysis.json + storyboard.json to exist)."""
    cfg = _load(config, model, None, None, fmt, width, height, aspect_ratio, False, no_resume, lora)
    from manhualizer.pipeline import run_render_only
    result = run_render_only(story, cfg)
    _console.print(f"[green]Render complete:[/green] {len(result.rendered_panels)} panel(s) → {result.output_dir}")


@app.command(name="models")
def list_models_cmd():
    """List all available image generation models and their capabilities."""
    from manhualizer.render import MODELS
    _console.print("[bold]Available models:[/bold]")
    for name, spec in MODELS.items():
        caps = spec.capabilities
        flags = []
        if caps.reference_images: flags.append("ref-images")
        if caps.lora:             flags.append("lora")
        if caps.negative_prompt:  flags.append("negative-prompt")
        flag_str = ", ".join(flags) or "text-prompt only"
        _console.print(f"  [cyan]{name:<16}[/cyan] {spec.display_name} ([dim]{flag_str}[/dim])")


def main():
    """Entry point for the manhualizer CLI."""
    app()

## Tests

In [ ]:
from typer.testing import CliRunner
from manhualizer.cli import app

runner = CliRunner()

# --help
result = runner.invoke(app, ["--help"])
assert result.exit_code == 0
assert "manhua" in result.output.lower()

# models subcommand
result = runner.invoke(app, ["models"])
assert result.exit_code == 0
assert "nanobanana" in result.output
assert "flux-klein" in result.output
assert "comfyui" in result.output

# analyze-only --help
result = runner.invoke(app, ["analyze-only", "--help"])
assert result.exit_code == 0

print("CLI tests OK")
print("\nModels output:")
print(runner.invoke(app, ["models"]).output)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()